# PACE IOP data

In [2]:
# imports
from importlib import reload
import os
import numpy as np

from matplotlib import pyplot as plt

import xarray
import pandas

from ocpy.pace import io as pace_io
from ocpy.utils import plotting
from ocpy.utils import coords as ocpy_coords
from ocpy.water import scattering

from bing.parameters import standard
from bing.models import utils as model_utils
from bing.priors import priors as bing_priors
from bing.fitting import inference as bing_inf
from bing import evaluate
from bing.fitting import chisq_fit
from bing import plotting as bing_plotting

# Locals
from grab_pace_granules import load_from_json
import fitting as m_fitting
import grab_pace_granules

# Load up

In [3]:
match_file = 'matched_argo_bgc_profiles_bbp.csv'
# Load up Argo profiles, already matched to PACE
matched = pandas.read_csv(match_file)

# Load up PACE granules
granules, pace = load_from_json('PACE_50clouds.json')


In [32]:
idx = 0

In [4]:
matched.pace_ids

0      PACE_OCI_L2_AOP_PACE_OCI.20240708T202028.L2.OC...
1      PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...
2      PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...
3      PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...
4      PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...
                             ...                        
807    PACE_OCI_L2_AOP_PACE_OCI.20250327T222202.L2.OC...
808    PACE_OCI_L2_AOP_PACE_OCI.20250209T163639.L2.OC...
809    PACE_OCI_L2_AOP_PACE_OCI.20250219T155847.L2.OC...
810    PACE_OCI_L2_AOP_PACE_OCI.20250301T152050.L2.OC...
811    PACE_OCI_L2_AOP_PACE_OCI.20250330T160134.L2.OC...
Name: pace_ids, Length: 812, dtype: object

In [11]:
matched

,cruise,filename,profile,lat,lon,time,solar_angle,pace_ids,closest_id,closest_file,closest_dist_km,closest_time,Bnw,Bnw_std,Bnw_lsig,Bnw_hsig,beta,aph
0,1902368,1902368QC.nc,13,56.348,-147.356,2024-07-09 04:35:00.000002048+00:00,13.134825,PACE_OCI_L2_AOP_PACE_OCI.20240708T202028.L2.OC...,PACE_OCI_L2_AOP_PACE_OCI.20240708T202028.L2.OC...,PACE_OCI.20240708T202028.L2.OC_AOP.V3_0.nc,671.392674,2024-07-08 20:22:57.500000+00:00,0.001467,0.000453,0.000075,0.000077,1.930553,0.041115
1,1902369,1902369QC.nc,4,45.006,-133.357,2024-05-09 22:07:00.000001536+00:00,58.276589,PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...,PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...,PACE_OCI.20240510T203409.L2.OC_AOP.V3_0.nc,0.620598,2024-05-10 20:36:38.500000+00:00,0.000501,0.000057,0.000037,0.000039,1.874987,0.005746
2,1902369,1902369QC.nc,5,45.020,-133.356,2024-05-10 09:02:59.999999488+00:00,-27.116450,PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...,PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...,PACE_OCI.20240510T203409.L2.OC_AOP.V3_0.nc,0.579841,2024-05-10 20:36:38.500000+00:00,0.000475,0.000124,0.000037,0.000037,1.868712,0.006338
3,1902369,1902369QC.nc,6,45.020,-133.354,2024-05-10 19:52:59.999998464+00:00,60.421381,PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...,PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...,PACE_OCI.20240510T203409.L2.OC_AOP.V3_0.nc,0.709744,2024-05-10 20:36:38.500000+00:00,0.000475,0.000126,0.000037,0.000037,1.868712,0.006338
4,1902369,1902369QC.nc,7,45.038,-133.348,2024-05-11 08:18:59.999995904+00:00,-26.535430,PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...,PACE_OCI_L2_AOP_PACE_OCI.20240510T203409.L2.OC...,PACE_OCI.20240510T203409.L2.OC_AOP.V3_0.nc,1.471472,2024-05-10 20:36:38.500000+00:00,0.000603,0.000123,0.000040,0.000043,1.817395,0.006617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
807,7902136,7902136QC.nc,13,3.838,-134.270,2025-03-28 02:37:00.000001536+00:00,6.590981,PACE_OCI_L2_AOP_PACE_OCI.20250327T222202.L2.OC...,PACE_OCI_L2_AOP_PACE_OCI.20250327T222202.L2.OC...,PACE_OCI.20250327T222202.L2.OC_AOP.V3_0.nc,1.083999,2025-03-27 22:24:31.500000+00:00,0.001520,0.000265,0.000057,0.000058,1.899728,0.018763
808,7902226,7902226QC.nc,3,27.225,-46.022,2025-02-08 18:16:00.000003072+00:00,29.864808,PACE_OCI_L2_AOP_PACE_OCI.20250209T163639.L2.OC...,PACE_OCI_L2_AOP_PACE_OCI.20250209T163639.L2.OC...,PACE_OCI.20250209T163639.L2.OC_AOP.V3_0.nc,7.557063,2025-02-09 16:39:08.500000+00:00,0.002266,0.000526,0.000131,0.000128,0.926795,0.006179
809,7902226,7902226QC.nc,4,27.479,-46.221,2025-02-18 20:26:59.999997440+00:00,6.099996,PACE_OCI_L2_AOP_PACE_OCI.20250219T155847.L2.OC...,PACE_OCI_L2_AOP_PACE_OCI.20250219T155847.L2.OC...,PACE_OCI.20250219T155847.L2.OC_AOP.V3_0.nc,0.406164,2025-02-19 16:01:16.500000+00:00,0.000950,0.000354,0.000036,0.000041,1.821743,0.003322
810,7902226,7902226QC.nc,5,27.563,-46.293,2025-02-28 21:48:59.999995904+00:00,-10.467588,PACE_OCI_L2_AOP_PACE_OCI.20250301T152050.L2.OC...,PACE_OCI_L2_AOP_PACE_OCI.20250301T152050.L2.OC...,PACE_OCI.20250301T152050.L2.OC_AOP.V3_0.nc,0.873742,2025-03-01 15:23:19.500000+00:00,0.000578,0.000100,0.000026,0.000028,1.927226,0.002177


In [36]:
imatched = matched.iloc[idx]
imatched.closest_id

'PACE_OCI_L2_AOP_PACE_OCI.20240708T202028.L2.OC_AOP.V3_0.nc_3.0'

## Load fits

In [37]:
fits_file = m_fitting.set_outfile(imatched)

In [41]:
fits = np.load(fits_file)
list(fits.keys())

['chains',
 'LM',
 'wave',
 'Rrs',
 'Rrs_sig',
 'Rrs_idx',
 'lon',
 'lat',
 'dist',
 'med',
 'p14',
 'p86',
 'model_names']

In [42]:
fits['Rrs_idx']

array([[1289,  153],
       [1288,  153],
       [1287,  153],
       [1290,  154],
       [1289,  154],
       [1288,  154],
       [1287,  154],
       [1289,  155],
       [1288,  155],
       [1287,  155]])

In [43]:
ix0, iy0 = fits['Rrs_idx'][0]
ix0, iy0

(np.int64(1289), np.int64(153))

## Load AOP

In [62]:
reload(pace_io)
aop_file = imatched.closest_file
aop_file = os.path.join(os.getenv('OS_COLOR'), 'PACE', 'L2_AOP', 
                     os.path.basename(aop_file))
os.path.exists(aop_file)
xds_aop, flags = pace_io.load_oci_l2(aop_file)

In [63]:
xds_aop

<xarray.Dataset> Size: 3GB
Dimensions:     (x: 1710, y: 1272, wl: 172)
Coordinates:
    latitude    (x, y) float32 9MB 41.94 41.96 41.98 42.01 ... 65.57 65.57 65.57
    longitude   (x, y) float32 9MB -134.7 -134.7 -134.6 ... -100.3 -100.1 -99.96
    wavelength  (wl) int32 688B 346 348 351 353 356 358 ... 712 713 714 717 719
Dimensions without coordinates: x, y, wl
Data variables:
    Rrs         (x, y, wl) float32 1GB -3.277e+04 -3.277e+04 ... -3.277e+04
    Rrs_unc     (x, y, wl) float32 1GB -3.277e+04 -3.277e+04 ... -3.277e+04
    FLH         (x, y) float32 9MB -3.277e+04 -3.277e+04 ... -3.277e+04
Attributes: (12/45)
    title:                             OCI Level-2 Data AOP
    product_name:                      PACE_OCI.20240708T202028.L2.OC_AOP.V3_...
    processing_version:                3.0
    history:                           l2gen par=/data5/sdpsoper/vdc/vpu24/wo...
    instrument:                        OCI
    platform:                          PACE
    ...                                ...
    geospatial_lon_max:                -99.95619
    geospatial_lon_min:                -150.1461
    startDirection:                    Ascending
    endDirection:                      Ascending
    day_night_flag:                    Day
    earth_sun_distance_correction:     0.9674713611602783

## Load IOPs

In [46]:
iop_file = imatched.closest_file.replace('AOP', 'IOP')
iop_file = iop_file.replace('V3_0', 'V3_1')
iop_file

'PACE_OCI.20240708T202028.L2.OC_IOP.V3_1.nc'

In [47]:
iop_file = os.path.join(os.getenv('OS_COLOR'), 'PACE', 'L2_IOP', 
                     iop_file)
iop_file

'/home/xavier/Projects/Oceanography/data/Color/PACE/L2_IOP/PACE_OCI.20240708T202028.L2.OC_IOP.V3_1.nc'

In [48]:
reload(pace_io)
xds_iop, flags = pace_io.load_iop_l2(iop_file)

## Examine

In [49]:
xds_iop

<xarray.Dataset> Size: 505MB
Dimensions:      (x: 1710, y: 1272, wl: 17)
Coordinates:
    latitude     (x, y) float32 9MB 41.94 41.96 41.98 ... 65.57 65.57 65.57
    longitude    (x, y) float32 9MB -134.7 -134.7 -134.6 ... -100.1 -99.96
    wavelength   (wl) int32 68B 400 413 425 442 460 475 ... 640 655 665 678 701
Dimensions without coordinates: x, y, wl
Data variables:
    a            (x, y, wl) float32 148MB -3.277e+04 -3.277e+04 ... -3.277e+04
    bb           (x, y, wl) float32 148MB -3.277e+04 -3.277e+04 ... -3.277e+04
    aph          (x, y, wl) float32 148MB -3.277e+04 -3.277e+04 ... -3.277e+04
    adg_s        (x, y) float32 9MB -3.277e+04 -3.277e+04 ... -3.277e+04
    adg_442      (x, y) float32 9MB -3.277e+04 -3.277e+04 ... -3.277e+04
    bbp_442      (x, y) float32 9MB -3.277e+04 -3.277e+04 ... -3.277e+04
    bbp_unc_442  (x, y) float32 9MB -2.8e+04 -2.8e+04 ... -2.8e+04 -2.8e+04
    bbp_s        (x, y) float32 9MB -3.277e+04 -3.277e+04 ... -3.277e+04
Attributes: (12/49)
    title:                             OCI Level-2 Data IOP
    product_name:                      PACE_OCI.20240708T202028.L2.OC_IOP.V3_...
    processing_version:                3.1
    history:                           l2gen par=/data8/sdpsoper/vdc/vpu27/wo...
    instrument:                        OCI
    platform:                          PACE
    ...                                ...
    geospatial_lon_min:                -150.1461
    startDirection:                    Ascending
    endDirection:                      Ascending
    day_night_flag:                    Day
    earth_sun_distance_correction:     0.9674713611602783
    geospatial_bounds:                 POLYGON ((-99.95620 65.57243, -150.146...

# Calculate bbp at 700nm


## $b_{b,p}(\lambda) = b_{b,p}(442) \, \frac{\lambda}{442\,\rm nm}^{bbp_s}$

In [50]:
xds_iop.bbp_442[ix0, iy0]

<xarray.DataArray 'bbp_442' ()> Size: 4B
array(0.00297001, dtype=float32)
Coordinates:
    latitude   float32 4B 56.68
    longitude  float32 4B -136.4

In [51]:
xds_iop.bbp_s[ix0, iy0]

<xarray.DataArray 'bbp_s' ()> Size: 4B
array(1.513135, dtype=float32)
Coordinates:
    latitude   float32 4B 56.68
    longitude  float32 4B -136.4

In [52]:
bbp_700 =  xds_iop.bbp_442[ix0, iy0] * (700./442.)**(-1*xds_iop.bbp_s[ix0, iy0])
bbp_700

<xarray.DataArray ()> Size: 4B
array(0.00148123, dtype=float32)
Coordinates:
    latitude   float32 4B 56.68
    longitude  float32 4B -136.4

# Our fit

In [ ]:
a_mean, bb_mean, a_5, a_95, bb_5, bb_95,\
            model_Rrs, sigRs = evaluate.reconstruct_from_chains(
            models, chains, perc=perc)